In [9]:
%pip install -q "qiskit==2.2.3" "numpy==2.2.6" "scipy==1.15.3" "matplotlib==3.10.0"
%pip install -q "torch==2.5.1" --index-url https://download.pytorch.org/whl/cpu

In [10]:
import sys
print(sys.version)
assert (3, 10) <= sys.version_info[:2] <= (3, 13), "Python 3.10~3.13 런타임을 선택하세요."

import numpy as np
values = np.array([8.5, 7.8, 6.5, 5.9, 8.9])
scaled = (values - values.min()) / np.ptp(values)
print(np.round(scaled, 4))
assert np.isclose(scaled[0], 2.6 / 3.0)

3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
[0.8667 0.6333 0.2    0.     1.    ]


In [15]:
import numpy as np
values = np.array([8.5, 7.8, 6.5, 5.9, 8.9])
scaled = (values - values.min()) / np.ptp(values)
print(np.round(scaled, 4))
assert np.isclose(scaled[0], 2.6 / 3.0)

[0.8667 0.6333 0.2    0.     1.    ]


In [16]:
import numpy as np
u = np.array([0.65, 0.85])
penalty = 2.0  # 람다: 선택 수 제약 벌점
n = len(u)
bits = ((np.arange(2**n)[:, None] >> np.arange(n)) & 1)
z = 1 - 2 * bits
cost = -(bits @ u) + penalty * (bits.sum(axis=1) - 1)**2
h = u / 2 - penalty * (n / 2 - 1)
J = penalty / 2
constant = -u.sum() / 2 + penalty * ((n / 2 - 1)**2 + n / 4)
ising = constant + z @ h + J * z[:, 0] * z[:, 1]
print("후보 순서의 비트:", bits.tolist())
print("QUBO:", cost)
print("Ising:", ising)
assert np.allclose(cost, ising)

후보 순서의 비트: [[0, 0], [1, 0], [0, 1], [1, 1]]
QUBO: [ 2.   -0.65 -0.85  0.5 ]
Ising: [ 2.   -0.65 -0.85  0.5 ]


In [17]:
import numpy as np
u = np.array([0.65, 0.85])
penalty = 2.0  # 람다: 선택 수 제약 벌점
n = len(u)
bits = ((np.arange(2**n)[:, None] >> np.arange(n)) & 1)
z = 1 - 2 * bits
cost = -(bits @ u) + penalty * (bits.sum(axis=1) - 1)**2
h = u / 2 - penalty * (n / 2 - 1)
J = penalty / 2
constant = -u.sum() / 2 + penalty * ((n / 2 - 1)**2 + n / 4)
ising = constant + z @ h + J * z[:, 0] * z[:, 1]
print("후보 순서의 비트:", bits.tolist())
print("QUBO:", cost)
print("Ising:", ising)
assert np.allclose(cost, ising)

후보 순서의 비트: [[0, 0], [1, 0], [0, 1], [1, 1]]
QUBO: [ 2.   -0.65 -0.85  0.5 ]
Ising: [ 2.   -0.65 -0.85  0.5 ]


In [18]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
u = np.array([0.65, 0.85])
penalty = 2.0
bits = ((np.arange(4)[:, None] >> np.arange(2)) & 1)
cost = -(bits @ u) + penalty * (bits.sum(axis=1) - 1)**2
h, J = u / 2, penalty / 2
constant = -u.sum() / 2 + penalty / 2
gamma, beta = 0.4, 0.3  # 감마: 비용 위상, 베타: 믹서 회전
qc = QuantumCircuit(2)
qc.h(range(2))
initial = Statevector.from_instruction(qc)
for i in range(2):
    qc.rz(2 * gamma * h[i], i)
qc.cx(0, 1)
qc.rz(2 * gamma * J, 1)
qc.cx(0, 1)
after_cost = Statevector.from_instruction(qc)
expected = initial.data * np.exp(-1j * gamma * (cost - constant))
assert np.allclose(after_cost.data, expected)
for i in range(2):
    qc.rx(2 * beta, i)
final = Statevector.from_instruction(qc)
for label, state in [("초기", initial), ("비용 회로 후", after_cost), ("믹서 후", final)]:
    print(label, "진폭:", np.round(state.data, 4), "확률:", np.round(state.probabilities(), 4))
print("기대 에너지:", float(final.probabilities() @ cost))

초기 진폭: [0.5+0.j 0.5+0.j 0.5+0.j 0.5+0.j] 확률: [0.25 0.25 0.25 0.25]
비용 회로 후 진폭: [0.3824-0.3221j 0.4679+0.1761j 0.4524+0.213j  0.4975-0.0499j] 확률: [0.25 0.25 0.25 0.25]
믹서 후 진폭: [0.4154-0.5494j 0.2825-0.1063j 0.267 -0.0694j 0.5305-0.2773j] 확률: [0.4745 0.0911 0.0761 0.3583]
기대 에너지: 1.0041884981428812


In [19]:
import time
from pathlib import Path
from importlib.metadata import version
import numpy as np
import torch
from torch import nn
from scipy.optimize import minimize
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print("패키지 버전:", {k: version(k) for k in ["qiskit", "numpy", "torch", "scipy", "matplotlib"]})
torch.set_num_threads(1)
torch.manual_seed(7)
started = time.perf_counter()
names = ["S1", "S2", "S3", "S4", "S5"]
features = np.array([
    [22, 4,  9, 0.80, 0.20, 0.85],  # S1
    [18, 6,  7, 0.75, 0.35, 0.90],  # S2
    [35, 8,  6, 0.60, 0.10, 0.70],  # S3
    [28, 5,  8, 0.82, 0.40, 0.88],  # S4
    [15, 3, 10, 0.77, 0.25, 0.95],  # S5
], dtype=np.float32)
labels = np.array([0.78, 0.82, 0.65, 0.80, 0.88], dtype=np.float32)

# 입력 열을 각각 0~1로 정규화합니다.
span = np.ptp(features, axis=0)
scaled = (features - features.min(axis=0)) / np.where(span == 0, 1, span)
X = torch.tensor(scaled, dtype=torch.float32)
y = torch.tensor(labels, dtype=torch.float32).reshape(-1, 1)
model = nn.Sequential(nn.Linear(X.shape[1], 16), nn.ReLU(),
                      nn.Linear(16, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
losses = []
for epoch in range(300):
    optimizer.zero_grad()
    loss = nn.functional.mse_loss(model(X), y)
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach()))
with torch.no_grad():
    u = model(X).numpy().reshape(-1).astype(float)
assert u.shape == (len(names),) and np.isfinite(u).all()

# lambda_pen: 람다, 정확히 한 곳 선택 제약의 벌점 계수
lambda_pen = 2.0
p = 1  # QAOA 반복 층수
shots = 4096
grid_size = 21
maxiter = 120
n = len(names)
# 행 번호 k의 i번째 열은 큐비트 i와 후보 i에 대응합니다.
bits = ((np.arange(2**n)[:, None] >> np.arange(n)) & 1)
feasible = bits.sum(axis=1) == 1
utility = bits @ u
qubo = -utility + lambda_pen * (bits.sum(axis=1) - 1)**2
# QUBO = lambda_pen + sum(a_i*x_i) + sum(b_ij*x_i*x_j), i<j
linear = -u - lambda_pen
pair = 2 * lambda_pen
z = 1 - 2 * bits
h = u / 2 - lambda_pen * (n / 2 - 1)
J = lambda_pen / 2
constant = -u.sum() / 2 + lambda_pen * ((n / 2 - 1)**2 + n / 4)
ising = constant + z @ h
for i in range(n):
    for j in range(i + 1, n):
        ising += J * z[:, i] * z[:, j]
assert np.allclose(qubo, ising, atol=1e-10)
print("QUBO–Ising 최대 오차:", float(np.max(np.abs(qubo - ising))))

# theta: 세타, 앞 p개는 감마, 뒤 p개는 베타입니다.
def circuit(theta):
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for layer in range(p):
        gamma = theta[layer]       # 감마: 비용 위상 변화량
        beta = theta[p + layer]    # 베타: 믹서 회전량
        for i in range(n):
            qc.rz(2 * gamma * h[i], i)
        for i in range(n):
            for j in range(i + 1, n):
                qc.cx(i, j)
                qc.rz(2 * gamma * J, j)
                qc.cx(i, j)
        for i in range(n):
            qc.rx(2 * beta, i)
    return qc

def energy(theta):
    sv = Statevector.from_instruction(circuit(theta))
    return float(sv.probabilities() @ qubo)

# p=1은 감마·베타의 21×21 조합을 평가합니다.
# p>=2는 같은 고정 시작점 3개에서 COBYLA로 2p개 각도를 탐색합니다.
assert p >= 1 and shots > 0 and grid_size >= 2
if p == 1:
    best_energy = float("inf")
    evaluations = 0
    for gamma in np.linspace(0, np.pi, grid_size):
        for beta in np.linspace(0, np.pi, grid_size):
            theta = np.array([gamma, beta])
            value = energy(theta)
            evaluations += 1
            if value < best_energy:
                best_energy, best_theta = value, theta.copy()
else:
    runs = []
    for gamma0, beta0 in [(0.2, 0.3), (0.7, 0.5), (1.2, 0.8)]:
        initial = np.r_[np.full(p, gamma0), np.full(p, beta0)]
        runs.append(minimize(energy, initial, method="COBYLA", options={"maxiter": maxiter}))
    result = min(runs, key=lambda r: r.fun)
    best_theta, best_energy = result.x, float(result.fun)
    evaluations = sum(r.nfev for r in runs)

sv = Statevector.from_instruction(circuit(best_theta))
probabilities = sv.probabilities()
assert np.isclose(probabilities.sum(), 1.0)
sv.seed(7)
counts = sv.sample_counts(shots)
observed = np.zeros(2**n, dtype=int)
for bitstring, count in counts.items():
    observed[int(bitstring, 2)] = count
assert observed.sum() == shots
# 전체 측정에서 최빈 상태를 선택하며 동률이면 정수 인덱스가 작은 상태를 선택합니다.
selected_index = int(np.argmax(observed))
x = bits[selected_index]
selected_names = [names[i] for i in range(n) if x[i]]
exact_utility = float(utility[feasible].max())
optimal = feasible & np.isclose(utility, exact_utility, atol=1e-8, rtol=0)
chosen_feasible = bool(feasible[selected_index])
print("후보 순서:", names)
for name, target, prediction in zip(names, labels, u):
    print(f"{name}: 교육용 목표={target:.3f}, 예측={prediction:.6f}")
print("최종 학습 MSE:", float(np.mean((u-labels)**2)))
print("감마:", best_theta[:p], "베타:", best_theta[p:])
print("회로 평가 횟수:", evaluations)
print("기대 에너지(상수항 포함):", best_energy)
print("선택 비트(후보 순서):", x.tolist(), "선택 항목:", selected_names)
print("선택 수:", int(x.sum()), "정확히 한 곳 선택:", chosen_feasible)
print("원래 목적함수(효용 합):", float(utility[selected_index]))
print("선택 상태 QUBO:", float(qubo[selected_index]))
print("정확 최적 효용:", exact_utility)
print("정확 최적 항목:", [names[i] for i in np.flatnonzero(np.isclose(u, exact_utility, atol=1e-8, rtol=0))])
print("최적값과 차이:", exact_utility - utility[selected_index] if chosen_feasible else "제약 위반: 비교 대상 제외")
print("실행 가능해 이론 확률 / 관측 비율:", float(probabilities[feasible].sum()), float(observed[feasible].sum()/shots))
print("최적해 이론 확률 / 관측 비율:", float(probabilities[optimal].sum()), float(observed[optimal].sum()/shots))
print("선택 상태 이론 확률 / 관측 비율:", probabilities[selected_index], observed[selected_index]/shots)
print("예산·대출·임대료 상한: 제약 미설정")

out = Path("day36_practice_3")
out.mkdir(exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), facecolor="white")
axes[0].plot(losses, color="darkorange")
axes[0].set(xlabel="Epoch", ylabel="MSE")
axes[1].bar(names, u, color="darkorange")
axes[1].set(xlabel="Candidate", ylabel="Predicted utility", ylim=(0, 1))
for i in np.flatnonzero(x):
    axes[2].plot(np.arange(features.shape[1]), scaled[i], marker="o", label=names[i])
axes[2].set(xlabel="Feature index (0-based)", ylabel="Normalized feature", ylim=(0, 1))
if selected_names:
    axes[2].legend()
fig.tight_layout()
fig.savefig(out / "results.png", dpi=150)
plt.close(fig)
np.savez(out / "results.npz", utility=u, bits=bits, probabilities=probabilities,
         observed=observed, selected=x, theta=best_theta)
print("결과 파일:", out / "results.png", out / "results.npz")
print("실행 시간(초):", round(time.perf_counter() - started, 2))

패키지 버전: {'qiskit': '2.2.3', 'numpy': '2.2.6', 'torch': '2.5.1+cpu', 'scipy': '1.15.3', 'matplotlib': '3.10.0'}
QUBO–Ising 최대 오차: 0.0
후보 순서: ['S1', 'S2', 'S3', 'S4', 'S5']
S1: 교육용 목표=0.780, 예측=0.782099
S2: 교육용 목표=0.820, 예측=0.820524
S3: 교육용 목표=0.650, 예측=0.649662
S4: 교육용 목표=0.800, 예측=0.798939
S5: 교육용 목표=0.880, 예측=0.878448
최종 학습 MSE: 1.6654331517429456e-06
감마: [0.15707963] 베타: [2.67035376]
회로 평가 횟수: 441
기대 에너지(상수항 포함): 0.3941227984341367
선택 비트(후보 순서): [0, 0, 0, 0, 0] 선택 항목: []
선택 수: 0 정확히 한 곳 선택: False
원래 목적함수(효용 합): 0.0
선택 상태 QUBO: 2.0
정확 최적 효용: 0.8784481287002563
정확 최적 항목: ['S5']
최적값과 차이: 제약 위반: 비교 대상 제외
실행 가능해 이론 확률 / 관측 비율: 0.5295839298305709 0.535888671875
최적해 이론 확률 / 관측 비율: 0.10733129254895796 0.105712890625
선택 상태 이론 확률 / 관측 비율: 0.120384504800079 0.116943359375
예산·대출·임대료 상한: 제약 미설정
결과 파일: day36_practice_3/results.png day36_practice_3/results.npz
실행 시간(초): 3.17


기본 설정에서는 선택 수 0인 상태가 가장 많이 관측되었습니다. 이 결과는 단일 선택 제약을 위반하므로 입지 결정을 확정할 수 없습니다. `p=2` 또는 `p=3`으로 실행하여 제약 충족 여부와 관측 비율을 확인합니다.

`최종 학습 MSE`는 5개 교육용 목표와 예측값 사이의 평균제곱오차입니다. `QUBO–Ising 최대 오차`는 부동소수점 반올림 수준이며 모든 32개 상태에 대해 두 비용이 일치해야 합니다.

`기대 에너지`는 전체 상태의 확률 가중 평균입니다. `원래 목적함수(효용 합)`은 실제 선택 비트에 해당하는 효용의 합입니다. `선택 상태 QUBO`에는 선택 수 위반 벌점이 포함됩니다. 각각의 값은 서로 다른 계산 대상을 설명합니다.

4096회 측정의 전체 최빈 상태를 결과로 사용합니다. 선택 수가 0개 또는 2개 이상이면 해당 비트와 항목을 그대로 출력하며 `정확히 한 곳 선택`은 False가 됩니다. 그 결과의 최적값 차이는 제약 위반으로 표시합니다. 실행 가능한 경우에는 정확 최적 효용에서 선택 효용을 뺀 차이를 출력합니다. 0에 가까울수록 해당 예측 점수 기준의 최적 선택에 가깝습니다.

`이론 확률`은 상태벡터에서 계산하고 `관측 비율`은 실제 샘플 횟수를 4096으로 나누어 계산합니다. 실행 가능해는 선택 수 1인 모든 상태이며, 최적해는 그중 효용이 가장 큰 상태입니다. 두 비율을 함께 읽으면 제약 충족과 최적 선택을 각각 평가할 수 있습니다. 완전탐색은 검증 기준을 제공하며 최종 선택은 측정 빈도로 결정됩니다.

`day36_practice_3/results.png`에서 학습 오차, 후보별 효용, 선택 후보의 정규화 특징을 확인합니다. 코랩 왼쪽 파일 메뉴에서 이미지를 열 수 있습니다. 특징 축 번호는 0부터 시작하며 입력 표의 열 순서와 같습니다. 정규화 값은 해당 특징의 상대적 크기이며 유리함의 방향은 각 열의 업무 의미를 따릅니다.